# Special relativity: Minkowski diagrams and Lorentz boosts

*astro · intermediate* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/astro/minkowski_diagram)

Events are 2-vectors $x^\mu = (t, x)$ and a change of inertial frame is one matrix $\Lambda^\mu{}_\nu$ applied to all of them with einsum. A twin flies to a star and back: the diagram in the home frame and in a boosted frame shows tilted lines of simultaneity, the invariant light cone, and the traveller's clock ticking $\int\sqrt{1 - v^2}\,dt$ — less than the twin who stayed home.

**Vector form**

$$\Delta s^2 = -c^2\Delta t^2 + \Delta x^2,\qquad t' = \gamma\,(t - vx/c^2),\;\; x' = \gamma\,(x - vt),\qquad \tau = \int\sqrt{1 - v^2/c^2}\;dt$$

**Index (tensor) form**

$$x'^\mu = \Lambda^\mu{}_\nu\,x^\nu,\qquad \Delta s^2 = \eta_{\mu\nu}\,\Delta x^\mu\Delta x^\nu,\qquad d\tau = \sqrt{-\eta_{\mu\nu}\,dx^\mu dx^\nu}$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $x^\mu$ | `events[..., :]` | (…, 2) | an event $(t, x)$, $\mu \in \{0, 1\}$, $c = 1$ |
| $\eta_{\mu\nu}$ | `metric()` | (2, 2) | Minkowski metric $\operatorname{diag}(-1, +1)$ |
| $\Lambda^\mu{}_\nu$ | `boost(beta)` | (…, 2, 2) | $\begin{pmatrix}\gamma & -\gamma\beta\\ -\gamma\beta & \gamma\end{pmatrix}$ |
| $\Lambda^\mu{}_\nu\,x^\nu$ | `einsum('mn,...n->...m', Lam, x)` | (…, 2) | repeated $\nu$ ⇒ summed: every event of a world line or grid boosted at once |
| $\eta_{\mu\nu}\Delta x^\mu\Delta x^\nu$ | `einsum('mn,...m,...n->...', eta, dx, dx)` | (…) | squared interval, the same in every frame |
| $\tau$ | `proper_time(events, eta)` | (N,) | running sum of $\sqrt{-\Delta s^2}$ along the world line |
| $\Lambda(\beta_1)\Lambda(\beta_2)$ | `boost(b1) @ boost(b2)` | (2, 2) | equals $\Lambda\big((\beta_1 + \beta_2)/(1 + \beta_1\beta_2)\big)$ |

Units: years and light-years, so $c = 1$ and light rays are the dashed 45° lines. The traveller flies at speed $v$ to a star at distance $L$, turns around instantly, and flies back. The amber line is the set of events the traveller calls "now": tilted by $v$ on the way out, it swings across the home twin's world line at the turnaround, which is where the "missing" years of the home twin go. The paradox is not symmetric because only the traveller changes inertial frame. Set the boost of the right panel equal to $v$ to sit in the traveller's outbound frame, or to $-v$ for the return leg. The light cone keeps its shape in every frame.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/astro/minkowski_diagram/main.py
"""Main loop: parameters → tensors → solve → plot."""

from pathlib import Path

import torch

from problems._common.runtime import cli

from . import equation, plot, solve, tensors


def run(params: dict, out_dir: Path) -> Path:
    # A handful of 2×2 matrices: everything stays on the CPU, in float64.
    v, distance, beta_view = params["v"], params["distance"], params["beta_view"]
    home, traveller = tensors.twin_world_lines(v, distance)
    T = float(home[-1, 0])
    grid = tensors.frame_grid(T + 2, -2 * T, 2 * T, step=1.0)

    sol = solve.solve(home, traveller, grid, beta_view)

    eta = equation.metric()
    reunion = traveller[-1] - traveller[0]
    s2 = equation.interval(reunion, eta)
    s2_boosted = equation.interval(equation.transform(sol["Lam"], reunion), eta)
    print(f"interval departure → reunion: Δs² = {s2:.6f} in S, {s2_boosted:.6f} in S′ (invariant)")
    b = torch.tensor(v, dtype=torch.float64)
    combined = equation.boost(b) @ equation.boost(b)
    print(
        f"Λ(v)Λ(v) = Λ({-combined[0, 1] / combined[0, 0]:.4f});  (v+v)/(1+v²) = {equation.velocity_addition(b, b):.4f}"
    )
    print(f"home twin ages {sol['tau_home'][-1]:.3f} yr, traveller ages {sol['tau_trav'][-1]:.3f} yr")
    print(f"expected traveller age 2L/v·√(1−v²) = {T * (1 - v**2) ** 0.5:.3f} yr")
    return plot.animate(sol, v, beta_view, out_dir / "minkowski_diagram.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/astro/minkowski_diagram/equation.py
r"""Physics → tensor translation.

Special relativity in one space dimension, with c = 1 (years and light-years). An event is a 4-vector
(here a 2-vector) x^μ = (t, x), μ ∈ {0, 1}. The Minkowski metric with signature (−, +) is

    \eta_{\mu\nu} = \operatorname{diag}(-1, +1)

and the squared interval between two events, the same for every inertial observer, is

    \Delta s^2 = \eta_{\mu\nu}\,\Delta x^\mu \Delta x^\nu = -\Delta t^2 + \Delta x^2      (μ and ν summed)

A boost to a frame moving with velocity β along x is the matrix

    \Lambda^\mu{}_\nu(\beta) = \begin{pmatrix} \gamma & -\gamma\beta \\ -\gamma\beta & \gamma \end{pmatrix},
    \qquad x'^\mu = \Lambda^\mu{}_\nu\, x^\nu,       γ = 1/\sqrt{1 - β^2}

Two boosts in a row are again a boost, Λ(β_1)Λ(β_2) = Λ((β_1 + β_2)/(1 + β_1β_2)) — relativistic velocity
addition. A clock carried along a world line ticks its proper time; for each short straight piece

    d\tau = \sqrt{-\eta_{\mu\nu}\,dx^\mu dx^\nu} = \sqrt{1 - v^2}\,dt .
"""

import torch


def metric(dtype=torch.float64, device=None) -> torch.Tensor:
    r"""η_{μν} = diag(−1, +1): shape (2, 2)."""
    return torch.diag(torch.tensor([-1.0, 1.0], dtype=dtype, device=device))


def lorentz_factor(beta: torch.Tensor) -> torch.Tensor:
    return 1 / torch.sqrt(1 - beta**2)


def boost(beta: torch.Tensor) -> torch.Tensor:
    r"""Λ^μ_ν(β) for any batch of speeds: (...,) → (..., 2, 2)."""
    beta = torch.as_tensor(beta, dtype=torch.float64)
    g = lorentz_factor(beta)
    row0 = torch.stack([g, -g * beta], dim=-1)
    row1 = torch.stack([-g * beta, g], dim=-1)
    return torch.stack([row0, row1], dim=-2)


def transform(Lam: torch.Tensor, x: torch.Tensor) -> torch.Tensor:
    r"""x'^μ = Λ^μ_ν x^ν for any batch of events (ν summed).   Lam: (2, 2), x: (..., 2) → (..., 2)"""
    return torch.einsum("mn,...n->...m", Lam, x)


def interval(dx: torch.Tensor, eta: torch.Tensor) -> torch.Tensor:
    r"""Δs² = η_{μν} Δx^μ Δx^ν (μ, ν summed).   dx: (..., 2) → (...)"""
    return torch.einsum("mn,...m,...n->...", eta, dx, dx)


def proper_time(events: torch.Tensor, eta: torch.Tensor) -> torch.Tensor:
    r"""τ along a world line given as consecutive events (N, 2): τ_k = Σ_{j<k} \sqrt{-Δs_j^2}, shape (N,)."""
    ds2 = interval(events.diff(dim=0), eta)  # (N-1,) negative for time-like steps
    dtau = torch.sqrt(torch.clamp(-ds2, min=0.0))
    return torch.cat([torch.zeros_like(dtau[:1]), dtau.cumsum(0)])


def velocity_addition(b1: torch.Tensor, b2: torch.Tensor) -> torch.Tensor:
    """(β1 + β2) / (1 + β1 β2): the speed of the combined boost."""
    return (b1 + b2) / (1 + b1 * b2)


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/astro/minkowski_diagram/tensors.py
"""Build the concrete tensors: the twins' world lines and a grid of reference lines, all as events x^μ = (t, x)."""

import torch


def twin_world_lines(v: float, distance: float, n: int = 801) -> tuple[torch.Tensor, torch.Tensor]:
    """Events (n, 2) along the home twin (x = 0) and the travelling twin, sampled at the same coordinate times.

    The traveller flies at speed v to a star at x = distance, turns around instantly and flies back.
    """
    T = 2 * distance / v  # coordinate time of the reunion
    t = torch.linspace(0.0, T, n, dtype=torch.float64)
    home = torch.stack([t, torch.zeros_like(t)], dim=-1)
    x = v * torch.minimum(t, T - t)  # out, then back
    traveller = torch.stack([t, x], dim=-1)
    return home, traveller


def frame_grid(t_max: float, x_min: float, x_max: float, step: float = 1.0, n: int = 50) -> torch.Tensor:
    """Lines of constant t and of constant x of the home frame, as events: (L, n, 2)."""
    lines = []
    for t0 in torch.arange(0.0, t_max + 1e-9, step, dtype=torch.float64):
        xs = torch.linspace(x_min, x_max, n, dtype=torch.float64)
        lines.append(torch.stack([torch.full_like(xs, t0), xs], dim=-1))
    for x0 in torch.arange(x_min, x_max + 1e-9, step, dtype=torch.float64):
        ts = torch.linspace(0.0, t_max, n, dtype=torch.float64)
        lines.append(torch.stack([ts, torch.full_like(ts, x0)], dim=-1))
    return torch.stack(lines)


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/astro/minkowski_diagram/solve.py
"""Proper times, clock ticks and lines of simultaneity — then boost everything into a moving frame."""

import torch

from .equation import boost, metric, proper_time, transform


def solve(home: torch.Tensor, traveller: torch.Tensor, grid: torch.Tensor, beta_view: float) -> dict:
    """Return the world lines in the home frame S and in the frame S' moving at beta_view, plus proper times."""
    eta = metric()
    Lam = boost(torch.tensor(beta_view))  # (2, 2)
    tau_home = proper_time(home, eta)  # (N,)
    tau_trav = proper_time(traveller, eta)  # (N,)
    return {
        "S": {"home": home, "traveller": traveller, "grid": grid},
        "S'": {"home": transform(Lam, home), "traveller": transform(Lam, traveller), "grid": transform(Lam, grid)},
        "tau_home": tau_home,
        "tau_trav": tau_trav,
        "Lam": Lam,
    }


def clock_ticks(events: torch.Tensor, tau: torch.Tensor, every: float = 1.0) -> torch.Tensor:
    """Events (K, 2) where the clock on this world line shows τ = every, 2·every, … (linear interpolation)."""
    marks = torch.arange(every, float(tau[-1]) + 1e-9, every, dtype=tau.dtype)
    k = torch.searchsorted(tau, marks).clamp(1, len(tau) - 1)
    f = ((marks - tau[k - 1]) / (tau[k] - tau[k - 1]))[:, None]
    return events[k - 1] + f * (events[k] - events[k - 1])


def simultaneity_line(event: torch.Tensor, v: float, half_length: float = 12.0) -> torch.Tensor:
    """Two events (2, 2) on the line of events that an observer at `event` moving with speed v calls "now".

    In the observer's rest frame "now" is t' = const; transformed back, that is the line t - t_e = v (x - x_e).
    """
    dx = torch.tensor([-half_length, half_length], dtype=event.dtype)
    return torch.stack([event[0] + v * dx, event[1] + dx], dim=-1)


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/astro/minkowski_diagram/plot.py
"""Two Minkowski diagrams of the twin trip: the home frame S and a boosted frame S'."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from problems._common.plotting import ACCENT, frame_indices, save_gif

from .equation import transform
from .solve import clock_ticks, simultaneity_line

HOME, TRAV, NOW = ACCENT[0], ACCENT[1], "#f59e0b"


def animate(sol: dict, v: float, beta_view: float, path: Path) -> Path:
    home_S, trav_S = sol["S"]["home"], sol["S"]["traveller"]
    tau_h, tau_t = sol["tau_home"], sol["tau_trav"]
    Lam = sol["Lam"]
    N = len(tau_h)
    T = float(home_S[-1, 0])

    ticks = {  # proper-time ticks every year, in S and in S'
        "home": clock_ticks(home_S, tau_h),
        "traveller": clock_ticks(trav_S, tau_t),
    }

    fig, axes = plt.subplots(1, 2, figsize=(8, 4.5))
    panels = []
    for ax, frame in zip(axes, ["S", "S'"]):
        data = sol[frame]
        world = np.concatenate([data["home"].numpy(), data["traveller"].numpy()])
        x_lo, x_hi = world[:, 1].min() - 1.5, world[:, 1].max() + 1.5
        t_hi = world[:, 0].max() + 0.5
        ax.set_xlim(x_lo, x_hi)
        ax.set_ylim(min(0.0, world[:, 0].min()) - 0.5, t_hi)
        ax.set_aspect("equal", adjustable="box")
        ax.grid(False)
        for line in data["grid"].numpy():  # the home frame's coordinate lines t = const, x = const
            ax.plot(line[:, 1], line[:, 0], color="#e5e7eb", lw=0.6, zorder=0)
        s = np.array([-40.0, 0.0, 40.0])  # light cone through the departure event: unchanged by any boost
        ax.plot(s, np.abs(s), color="#9ca3af", ls=(0, (4, 3)), lw=1)
        ax.set_xlabel(f"x{'′' if frame != 'S' else ''} (light-years)")
        ax.set_ylabel(f"t{'′' if frame != 'S' else ''} (years)")
        mats = Lam if frame != "S" else torch.eye(2, dtype=torch.float64)
        tick_h = transform(mats, ticks["home"]).numpy()
        tick_t = transform(mats, ticks["traveller"]).numpy()
        panels.append(
            {
                "ax": ax,
                "mats": mats,
                "home": data["home"].numpy(),
                "trav": data["traveller"].numpy(),
                "tick_h": tick_h,
                "tick_t": tick_t,
                "line_h": ax.plot([], [], color=HOME, lw=2, label="home twin")[0],
                "line_t": ax.plot([], [], color=TRAV, lw=2, label="traveller")[0],
                "dots_h": ax.plot([], [], "o", ms=3.5, color=HOME)[0],
                "dots_t": ax.plot([], [], "o", ms=3.5, color=TRAV)[0],
                "now": ax.plot([], [], color=NOW, lw=1.5, label="traveller's “now”")[0],
            }
        )
    axes[0].set_title("home frame S   (dots: one year of proper time)", fontsize=9)
    axes[1].set_title(f"frame S′ boosted by β = {beta_view:+.2f}:  x′ = Λx", fontsize=9)
    axes[0].legend(fontsize=7, loc="lower right", framealpha=0.95)
    ages = fig.text(0.5, 0.01, "", ha="center", fontsize=9)

    def update(k: int):
        t_now = float(home_S[k, 0])
        v_now = v if t_now < T / 2 else -v
        now_S = simultaneity_line(trav_S[k], v_now)
        for p in panels:
            p["line_h"].set_data(p["home"][: k + 1, 1], p["home"][: k + 1, 0])
            p["line_t"].set_data(p["trav"][: k + 1, 1], p["trav"][: k + 1, 0])
            n_h, n_t = int(tau_h[k]), int(tau_t[k] + 1e-9)
            p["dots_h"].set_data(p["tick_h"][:n_h, 1], p["tick_h"][:n_h, 0])
            p["dots_t"].set_data(p["tick_t"][:n_t, 1], p["tick_t"][:n_t, 0])
            now = transform(p["mats"], now_S).numpy()
            p["now"].set_data(now[:, 1], now[:, 0])
        ages.set_text(
            f"t = {t_now:4.1f} yr    home twin aged {tau_h[k]:4.2f} yr    traveller aged {tau_t[k]:4.2f} yr"
            f"    (∫√(1−v²) dt)"
        )

    fig.tight_layout(rect=(0, 0.05, 1, 1))
    frames = frame_indices(N, 90) + [N - 1] * 15  # hold the reunion for a moment
    return save_gif(fig, update, frames, path)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'v': 0.8,
    'distance': 4.0,
    'beta_view': 0.8
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.astro.minkowski_diagram")]:
    del sys.modules[name]
main = importlib.import_module("problems.astro.minkowski_diagram.main")
problem_dir = Path("problems/astro/minkowski_diagram")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
